# Ablation Study: Architectural Components and Design Decisions

**Project:** Comparative Analysis of CNN, RNN, and Hybrid CNN–RNN Architectures for Papilledema Detection and Severity Grading  
**Focus Sections:**
1. **CNN Comparison** (ResNet50 vs. DenseNet121 vs. EfficientNet-B0)
2. **RNN Directionality Comparison** (Unidirectional LSTM vs. Bidirectional LSTM)
3. **Hybrid CNN–RNN Comparison** (CNN Baselines vs. ResNet50+LSTM vs. ResNet50+BiLSTM)
4. **OCT Sequence Length Comparison** (Window Lengths $L \in \{4, 8, 12\}$)


In [ ]:
import os
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

out_dir = '../results/tables/consolidated_5fold_10fold'
os.makedirs(out_dir, exist_ok=True)

df5_agg = pd.read_csv('../results/tables/documentation_tables/D01_cross_validation_5fold.csv')
df10_agg = pd.read_csv('../results/cross_validation/10_fold/cv_10fold_models_aggregated.csv')

track_map = {
    'fundus_3class': 'Fundus',
    'oct_individual': 'Individual OCT',
    'oct_sequence': 'OCT Sequence',
    'oct_severity': 'OCT Severity'
}
print("Setup complete. Aggregated model data loaded.")

Setup complete. Aggregated model data loaded.


## 1. CNN Comparison

Evaluating three primary convolutional feature extractors across standard transfer-learning configurations under both 5-Fold and 10-Fold CV:
- **ResNet50:** Deep residual architecture with additive identity skips.
- **DenseNet121:** Dense connectivity concatenating feature maps from all preceding layers.
- **EfficientNet-B0:** Compound-scaled depthwise separable inverted bottlenecks.

In [ ]:
cnn_backbones = ['resnet50', 'densenet121', 'efficientnet_b0']
cnn_tracks = ['fundus_3class', 'oct_individual', 'oct_severity']
t3_rows = []
for t_key in cnn_tracks:
    for m in cnn_backbones:
        v5 = df5_agg[(df5_agg['track']==t_key) & (df5_agg['model']==m)]['macro_f1_mean'].values[0]
        v10 = df10_agg[(df10_agg['track']==t_key) & (df10_agg['model']==m)]['macro_f1_mean'].values[0]
        t3_rows.append({
            'Experimental Track': track_map[t_key],
            'Model': m,
            '5-Fold Macro-F1': round(v5, 4),
            '10-Fold Macro-F1': round(v10, 4)
        })
table_3 = pd.DataFrame(t3_rows)
table_3.to_csv(os.path.join(out_dir, 'table3_ablation_cnn_backbone.csv'), index=False)
display(table_3)

  Experimental Track            Model  5-Fold Macro-F1  10-Fold Macro-F1
0             Fundus         resnet50           0.9538            0.9547
1             Fundus      densenet121           0.9613            0.9657
2             Fundus  efficientnet_b0           0.9592            0.9758
3     Individual OCT         resnet50           0.8895            0.8647
4     Individual OCT      densenet121           0.9621            0.9616
5     Individual OCT  efficientnet_b0           0.9050            0.9157
6       OCT Severity         resnet50           0.6669            0.8257
7       OCT Severity      densenet121           0.7601            0.8808
8       OCT Severity  efficientnet_b0           0.5998            0.8257


## 2. RNN Directionality Comparison

Evaluating unidirectional LSTM versus Bidirectional LSTM (BiLSTM):

> **Empirical Findings:**
> - In **5-Fold Cross-Validation**, BiLSTM outperforms LSTM across **all four experimental tracks**.
> - In **10-Fold Cross-Validation**, LSTM outperforms BiLSTM in **three of four tracks** (Fundus, Individual OCT, and OCT Sequence), while BiLSTM outperforms LSTM in **OCT Severity** ($0.3223 > 0.3001$).
> - **Neither architecture possesses universal superiority.** Standalone BiLSTM doubles the recurrent parameter count, which causes mild overfitting on smaller validation fold partitions in 10-fold CV.

In [ ]:
rnn_models = ['lstm', 'bilstm']
t4_rows = []
for t_key in track_map.keys():
    for m in rnn_models:
        v5 = df5_agg[(df5_agg['track']==t_key) & (df5_agg['model']==m)]['macro_f1_mean'].values[0]
        v10 = df10_agg[(df10_agg['track']==t_key) & (df10_agg['model']==m)]['macro_f1_mean'].values[0]
        t4_rows.append({
            'Experimental Track': track_map[t_key],
            'Model': m.upper(),
            '5-Fold Macro-F1': round(v5, 4),
            '10-Fold Macro-F1': round(v10, 4)
        })
table_4 = pd.DataFrame(t4_rows)
table_4.to_csv(os.path.join(out_dir, 'table4_ablation_rnn_directionality.csv'), index=False)
display(table_4)

  Experimental Track   Model  5-Fold Macro-F1  10-Fold Macro-F1
0             Fundus    LSTM           0.7483            0.7472
1             Fundus  BILSTM           0.7500            0.7293
2     Individual OCT    LSTM           0.4963            0.4258
3     Individual OCT  BILSTM           0.5245            0.3953
4       OCT Sequence    LSTM           0.4047            0.5344
5       OCT Sequence  BILSTM           0.4175            0.4700
6       OCT Severity    LSTM           0.1588            0.3001
7       OCT Severity  BILSTM           0.2222            0.3223


## 3. Hybrid CNN–RNN Comparison

Controlled ablation evaluating the impact of appending recurrent layers to convolutional feature extractors:

> **Empirical Findings:**
> 1. **5-Fold CV:** ResNet50+LSTM performs **below** pure ResNet50 for Fundus ($0.9535 < 0.9538$) and Individual OCT ($0.8712 < 0.8895$). Only ResNet50+BiLSTM outperforms ResNet50 across all 5-fold tracks.
> 2. **10-Fold CV:** Both ResNet50+LSTM and ResNet50+BiLSTM outperform pure ResNet50 in all image tracks, but **ResNet50+LSTM outperforms ResNet50+BiLSTM** in Individual OCT ($0.9237 > 0.9022$) and OCT Sequence ($0.8732 > 0.8337$).
> 3. **Standalone CNN Superiority:** DenseNet121 pure CNN ($0.9621$ / $0.9616$) outperforms all ResNet-50 hybrids in Individual OCT across both protocols.

In [ ]:
t5_rows = [
    # Fundus
    {'Experimental Track': 'Fundus', 'Configuration': 'CNN baseline (DenseNet121)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='fundus_3class')&(df5_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='fundus_3class')&(df10_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Fundus', 'Configuration': 'ResNet50 (Matched CNN Baseline)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='fundus_3class')&(df5_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='fundus_3class')&(df10_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Fundus', 'Configuration': 'ResNet50 + LSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='fundus_3class')&(df5_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='fundus_3class')&(df10_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Fundus', 'Configuration': 'ResNet50 + BiLSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='fundus_3class')&(df5_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='fundus_3class')&(df10_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4)},
    # Individual OCT
    {'Experimental Track': 'Individual OCT', 'Configuration': 'CNN baseline (DenseNet121)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_individual')&(df5_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_individual')&(df10_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Individual OCT', 'Configuration': 'ResNet50 (Matched CNN Baseline)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_individual')&(df5_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_individual')&(df10_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Individual OCT', 'Configuration': 'ResNet50 + LSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_individual')&(df5_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_individual')&(df10_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'Individual OCT', 'Configuration': 'ResNet50 + BiLSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_individual')&(df5_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_individual')&(df10_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4)},
    # OCT Sequence
    {'Experimental Track': 'OCT Sequence', 'Configuration': 'ResNet50 + LSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_sequence')&(df5_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_sequence')&(df10_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'OCT Sequence', 'Configuration': 'ResNet50 + BiLSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_sequence')&(df5_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_sequence')&(df10_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4)},
    # OCT Severity
    {'Experimental Track': 'OCT Severity', 'Configuration': 'CNN baseline (DenseNet121)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_severity')&(df5_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_severity')&(df10_agg['model']=='densenet121')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'OCT Severity', 'Configuration': 'ResNet50 (Matched CNN Baseline)', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_severity')&(df5_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_severity')&(df10_agg['model']=='resnet50')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'OCT Severity', 'Configuration': 'ResNet50 + LSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_severity')&(df5_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_severity')&(df10_agg['model']=='resnet50_lstm')]['macro_f1_mean'].values[0], 4)},
    {'Experimental Track': 'OCT Severity', 'Configuration': 'ResNet50 + BiLSTM', '5-Fold Macro-F1': round(df5_agg[(df5_agg['track']=='oct_severity')&(df5_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4), '10-Fold Macro-F1': round(df10_agg[(df10_agg['track']=='oct_severity')&(df10_agg['model']=='resnet50_bilstm')]['macro_f1_mean'].values[0], 4)},
]
table_5 = pd.DataFrame(t5_rows)
table_5.to_csv(os.path.join(out_dir, 'table5_ablation_hybrid_architecture.csv'), index=False)
display(table_5)

   Experimental Track                    Configuration  5-Fold Macro-F1  10-Fold Macro-F1
0              Fundus       CNN baseline (DenseNet121)           0.9613            0.9657
1              Fundus  ResNet50 (Matched CNN Baseline)           0.9538            0.9547
2              Fundus                  ResNet50 + LSTM           0.9535            0.9629
3              Fundus                ResNet50 + BiLSTM           0.9624            0.9673
4      Individual OCT       CNN baseline (DenseNet121)           0.9621            0.9616
5      Individual OCT  ResNet50 (Matched CNN Baseline)           0.8895            0.8647
6      Individual OCT                  ResNet50 + LSTM           0.8712            0.9237
7      Individual OCT                ResNet50 + BiLSTM           0.9193            0.9022
8        OCT Sequence                  ResNet50 + LSTM           0.8945            0.8732
9        OCT Sequence                ResNet50 + BiLSTM           0.8984            0.8337
10       O

## 4. OCT Sequence Length Comparison

Evaluating sequence window lengths $L \in \{4, 8, 12\}$:
- **Window-Level:** Evaluated across $N = 146$ sliding window segments of length $L$.
- **Sequence-Level:** Evaluated across $N = 21$ whole patient scan volumes via majority consensus voting.

In [ ]:
table_6 = pd.DataFrame([
    {'Sequence Length': 4, 'Evaluation Protocol': '5-Fold CV', 'Window-Level Macro-F1': 0.8820, 'Sequence-Level Accuracy': '95.24%', 'Correct Sequences': '20 / 21'},
    {'Sequence Length': 4, 'Evaluation Protocol': '10-Fold CV', 'Window-Level Macro-F1': 'Unavailable (L=8 default)', 'Sequence-Level Accuracy': 'Unavailable', 'Correct Sequences': 'Unavailable'},
    {'Sequence Length': 8, 'Evaluation Protocol': '5-Fold CV', 'Window-Level Macro-F1': 0.8984, 'Sequence-Level Accuracy': '90.48%', 'Correct Sequences': '19 / 21'},
    {'Sequence Length': 8, 'Evaluation Protocol': '10-Fold CV', 'Window-Level Macro-F1': 0.8337, 'Sequence-Level Accuracy': 'Evaluated at Window-level', 'Correct Sequences': 'N/A (Window-level)'},
    {'Sequence Length': 12, 'Evaluation Protocol': '5-Fold CV', 'Window-Level Macro-F1': 0.8903, 'Sequence-Level Accuracy': '95.24%', 'Correct Sequences': '20 / 21'},
    {'Sequence Length': 12, 'Evaluation Protocol': '10-Fold CV', 'Window-Level Macro-F1': 'Unavailable (L=8 default)', 'Sequence-Level Accuracy': 'Unavailable', 'Correct Sequences': 'Unavailable'},
])
table_6.to_csv(os.path.join(out_dir, 'table6_ablation_oct_sequence_length.csv'), index=False)
display(table_6)

   Sequence Length Evaluation Protocol      Window-Level Macro-F1    Sequence-Level Accuracy   Correct Sequences
0                4           5-Fold CV                     0.8820                     95.24%             20 / 21
1                4          10-Fold CV  Unavailable (L=8 default)                Unavailable         Unavailable
2                8           5-Fold CV                     0.8984                     90.48%             19 / 21
3                8          10-Fold CV                     0.8337  Evaluated at Window-level  N/A (Window-level)
4               12           5-Fold CV                     0.8903                     95.24%             20 / 21
5               12          10-Fold CV  Unavailable (L=8 default)                Unavailable         Unavailable
